# Mutation → Disease Relation Pipeline

Builds a unified, deduplicated edge table for the **Mutation–Disease** relation.

**Output schema:** `head | relation | tail | head_type | relation_type | tail_type | kg_source | kg_type | head_id_is | tail_id_is | head_detail_name | tail_detail_name`

---
**Sources included:**
- HALD (Mutation_Disease_HALD - Native Aging)
- Monarch (Human_SequenceVariant_Disease_Monarch - Native Generalised)

**Base paths:**
- KG data: `/storage/Arushi/090526_EvoAge/kg_formation/`
- Processed KG files: `/storage/Arushi/090526_EvoAge/kg_formation/processed_data/`

## 0 · Imports & Base Paths

In [1]:
import pandas as pd
import os

BASE_DIR  = '/storage/Arushi/090526_EvoAge/kg_formation/'
PROC_DIR  = BASE_DIR + 'processed_data/'
INPUT_DIR = BASE_DIR + 'data_to_relationwise_triples_sorting/4EvoAGE/'
OUT_PATH  = BASE_DIR + 'processed_data_relation_wise_merge/generalised/MUTATION_DISEASE/ALL_MUTATION_DISEASE.csv'
DB_DIR     = BASE_DIR + 'data_collection/databases_for_mapping/'

REQUIRED_COLS = [
    'head', 'relation', 'tail', 'head_type', 'relation_type', 'tail_type',
    'kg_source', 'kg_type', 'head_id_is', 'tail_id_is', 'head_detail_name', 'tail_detail_name'
]



## DO ids

In [3]:
DO_All_File = pd.read_csv(DB_DIR + 'DO/All_DO_ref_files.csv')
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO dict: {len(DOID_disname_dict):,} entries")

DO dict: 11,804 entries


In [4]:
MedGen_CUID_Source_ID = pd.read_csv(DB_DIR + 'MESH/MeSH/MedGenIDMappings.txt', sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id', 'source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))

MONDO_2_MESH = MedGen_CUID_Source_ID[MedGen_CUID_Source_ID['source_id'].str.contains('MONDO', na=False)]
MONDO_2_MESH_dict    = dict(zip(MONDO_2_MESH['source_id'],     MONDO_2_MESH['#CUI_or_CN_id']))
MONDOMESH_2_des_dict = dict(zip(MONDO_2_MESH['#CUI_or_CN_id'], MONDO_2_MESH['pref_name']))
print(f"MONDO→MESH: {len(MONDO_2_MESH_dict):,} entries")

MONDO→MESH: 22,703 entries


In [5]:
Mesh2DOID = pd.read_csv(DB_DIR + 'DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv', sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict = dict(zip(Mesh2DOID['xrefs'], Mesh2DOID['id']))

MESH = pd.read_csv(DB_DIR + 'MESH/MESH_Combined.csv')
MESH_dict = dict(zip(MESH['ID'], MESH['Name']))

Mesh_supp = pd.read_csv(DB_DIR + 'MESH/Mesh_Supplementary_Info.csv')
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))

print(f"MESH dict: {len(MESH_dict):,} | MESH supp: {len(Mesh_supp_dict):,}")

MESH dict: 38,520 | MESH supp: 324,150


In [6]:
import numpy as np
def resolve_disease_tail(df):
    """Map MONDO→MESH, derive tail_detail_name, assign tail_id_is."""
    # Map MONDO IDs to MESH, fallback to original
    df['tail'] = df['tail'].map(MONDO_2_MESH_dict).fillna(df['tail'])
    
    # Drop rows still starting with MONDO
    df = df[~df['tail'].astype(str).str.startswith('MONDO')].copy()
    
    # Derive tail_detail_name from dictionaries
    df['tail_detail_name'] = df['tail'].apply(
        lambda x: DOID_disname_dict.get(x)
                  if isinstance(x, str) and x.startswith('DOID')
                  else (MESH_dict.get(x) or Mesh_supp_dict.get(x) or MONDOMESH_2_des_dict.get(x))
    )
    
    # Assign tail_id_is based on prefix
    df['tail_id_is'] = np.where(
        df['tail'].isna(), None,
        np.where(df['tail'].astype(str).str.startswith('DOID'), 'DOID', 'MESH')
    )
    
    return df

## 1 · Load Source Files

In [8]:
# 1. HALD Mutation Disease Native (Aging)
Hald_Mut_Dis = pd.read_csv(PROC_DIR + 'hald/Mutation_Disease_HALD.csv')
Hald_Mut_Dis.columns = Hald_Mut_Dis.columns.str.lower()

if 'relation_source' in Hald_Mut_Dis.columns:
    Hald_Mut_Dis = Hald_Mut_Dis.rename(columns={'relation_source': 'kg_source'})
if 'kgsource' in Hald_Mut_Dis.columns:
    Hald_Mut_Dis = Hald_Mut_Dis.rename(columns={'kgsource': 'kg_source'})
if 'head_name' in Hald_Mut_Dis.columns:
    Hald_Mut_Dis = Hald_Mut_Dis.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in Hald_Mut_Dis.columns:
    Hald_Mut_Dis = Hald_Mut_Dis.rename(columns={'tail_name': 'tail_detail_name'})

Hald_Mut_Dis['head_id_is'] = 'dbSNP'
Hald_Mut_Dis['relation'] = 'Mutation_Disease'
Hald_Mut_Dis['head_type'] = 'Mutation'
Hald_Mut_Dis['tail_type'] = 'Disease'
Hald_Mut_Dis['kg_type'] = 'Aging'

if 'head_detail_name' not in Hald_Mut_Dis.columns:
    Hald_Mut_Dis['head_detail_name'] = Hald_Mut_Dis['head']
else:
    Hald_Mut_Dis['head_detail_name'] = Hald_Mut_Dis['head_detail_name'].fillna(Hald_Mut_Dis['head'])


Hald_Mut_Dis

,head,relation,tail,head_type,relation_type,tail_type,kg_source,tail_detail_name,tail_id_is,head_id_is,kg_type,head_detail_name
0,rs4612666,Mutation_Disease,DOID:824,Mutation,implicated,Disease,HALD_KG,Periodontitis,DOID,dbSNP,Aging,rs4612666
1,rs1143634,Mutation_Disease,DOID:824,Mutation,implicated,Disease,HALD_KG,Periodontitis,DOID,dbSNP,Aging,rs1143634
2,rs2619112,Mutation_Disease,D050723,Mutation,associated,Disease,HALD_KG,"Fractures, Bone",MESH,dbSNP,Aging,rs2619112
3,rs2942168,Mutation_Disease,D010302,Mutation,associated,Disease,HALD_KG,"Parkinson Disease, Secondary",MESH,dbSNP,Aging,rs2942168
4,rs2942168,Mutation_Disease,D010300,Mutation,associated,Disease,HALD_KG,Parkinson Disease,MESH,dbSNP,Aging,rs2942168
...,...,...,...,...,...,...,...,...,...,...,...,...
629,rs104893736,Mutation_Disease,DOID:83,Mutation,associated,Disease,HALD_KG,Cataract,DOID,dbSNP,Aging,rs104893736
630,rs120074195,Mutation_Disease,D001145,Mutation,associated,Disease,HALD_KG,"Arrhythmias, Cardiac",MESH,dbSNP,Aging,rs120074195
631,rs120074195,Mutation_Disease,D001145,Mutation,incorporated,Disease,HALD_KG,"Arrhythmias, Cardiac",MESH,dbSNP,Aging,rs120074195
632,rs104893736,Mutation_Disease,DOID:83,Mutation,cause,Disease,HALD_KG,Cataract,DOID,dbSNP,Aging,rs104893736


In [11]:
# 2. Monarch SequenceVariant Disease Native (Generalised)
Monarch = pd.read_csv(PROC_DIR + 'Monarch/Human/Human_SequenceVariant_Disease_Monarch.csv')
Monarch.columns = Monarch.columns.str.lower()

if 'relation_source' in Monarch.columns:
    Monarch = Monarch.rename(columns={'relation_source': 'kg_source'})
if 'kgsource' in Monarch.columns:
    Monarch = Monarch.rename(columns={'kgsource': 'kg_source'})
if 'head_name' in Monarch.columns:
    Monarch = Monarch.rename(columns={'head_name': 'head_detail_name'})
if 'tail_name' in Monarch.columns:
    Monarch = Monarch.rename(columns={'tail_name': 'tail_detail_name'})

Monarch['head_id_is'] = 'dbSNP'
Monarch['relation'] = 'Mutation_Disease'
Monarch['head_type'] = 'Mutation'
Monarch['tail_type'] = 'Disease'
Monarch['kg_type'] = 'Generalised'
Monarch= resolve_disease_tail(Monarch)
Monarch

,head,tail,relation_type,kg_source,kg_source,head_detail_name,head_type,head_id_is,head_taxon,head_taxon_label,...,head_type_clean,tail_type_clean,relation,species_species,head_name_protein,head_name_mutaion,head_name_protein_uniprot_mapped,head_orignal,tail_do_name,kg_type
0,MTOR_p.Leu2427Pro,CN300503,causes,infores:clinvar,MonarchKG,NM_004958.4(MTOR):c.7280T>C (p.Leu2427Pro),Mutation,dbSNP,9606.0,Homo sapiens,...,SequenceVariant,Disease,Mutation_Disease,Homo sapiens_Homo sapiens,MTOR,p.Leu2427Pro,P42345,CLINVAR:417723,MONDO:0100283,Generalised
1,MTOR_p.Ser2215Phe,CN300503,causes,infores:clinvar,MonarchKG,NM_004958.3(MTOR):c.6644C>T (p.Ser2215Phe),Mutation,dbSNP,9606.0,Homo sapiens,...,SequenceVariant,Disease,Mutation_Disease,Homo sapiens_Homo sapiens,MTOR,p.Ser2215Phe,P42345,CLINVAR:156703,MONDO:0100283,Generalised
2,MTOR_p.Ser2215Tyr,CN300503,causes,infores:clinvar,MonarchKG,NM_004958.4(MTOR):c.6644C>A (p.Ser2215Tyr),Mutation,dbSNP,9606.0,Homo sapiens,...,SequenceVariant,Disease,Mutation_Disease,Homo sapiens_Homo sapiens,MTOR,p.Ser2215Tyr,P42345,CLINVAR:376129,MONDO:0100283,Generalised
3,NM_004958.3:c.5930C>G,CN300503,causes,infores:clinvar,MonarchKG,NM_004958.3:c.5930C>G,Mutation,dbSNP,9606.0,Homo sapiens,...,SequenceVariant,Disease,Mutation_Disease,Homo sapiens_Homo sapiens,NaN,NaN,NaN,CLINVAR:1296989,MONDO:0100283,Generalised
4,MTOR_p.Thr1977Lys,CN300503,causes,infores:clinvar,MonarchKG,NM_004958.3(MTOR):c.5930C>A (p.Thr1977Lys),Mutation,dbSNP,9606.0,Homo sapiens,...,SequenceVariant,Disease,Mutation_Disease,Homo sapiens_Homo sapiens,MTOR,p.Thr1977Lys,P42345,CLINVAR:156702,MONDO:0100283,Generalised
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14644,LDLR_p.His583Asp,C0745103,associated_with_increased_likelihood_of,infores:clingen,MonarchKG,NM_000527.5(LDLR):c.1747C>G (p.His583Asp),Mutation,dbSNP,9606.0,Homo sapiens,...,SequenceVariant,Disease,Mutation_Disease,Homo sapiens_Homo sapiens,LDLR,p.His583Asp,P01130,CLINVAR:183124,MONDO:0007750,Generalised
14645,LDLR_p.His583Arg,C0745103,associated_with_increased_likelihood_of,infores:clingen,MonarchKG,NM_000527.5(LDLR):c.1748A>G (p.His583Arg),Mutation,dbSNP,9606.0,Homo sapiens,...,SequenceVariant,Disease,Mutation_Disease,Homo sapiens_Homo sapiens,LDLR,p.His583Arg,P01130,CLINVAR:252012,MONDO:0007750,Generalised
14646,LDLR_p.His583Gln,C0745103,genetically_associated_with,infores:clingen,MonarchKG,NM_000527.5(LDLR):c.1749C>G (p.His583Gln),Mutation,dbSNP,9606.0,Homo sapiens,...,SequenceVariant,Disease,Mutation_Disease,Homo sapiens_Homo sapiens,LDLR,p.His583Gln,P01130,CLINVAR:441220,MONDO:0007750,Generalised
14647,LDLR_p.His583Gln,C0745103,associated_with_increased_likelihood_of,infores:clingen,MonarchKG,NM_000527.5(LDLR):c.1749C>A (p.His583Gln),Mutation,dbSNP,9606.0,Homo sapiens,...,SequenceVariant,Disease,Mutation_Disease,Homo sapiens_Homo sapiens,LDLR,p.His583Gln,P01130,CLINVAR:252014,MONDO:0007750,Generalised


## 2 · Consolidate

In [12]:
source_dfs = [ Hald_Mut_Dis, Monarch]
aligned = []
for df in source_dfs:
    df = df.copy()
    df = df.loc[:, ~df.columns.duplicated()]
    for col in REQUIRED_COLS:
        if col not in df.columns:
            df[col] = None       
    aligned.append(df[REQUIRED_COLS])
final_df = pd.concat(aligned, ignore_index=True)
final_df = final_df[final_df['head'].astype(str).str.upper() != 'NAN']
final_df

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name
0,rs4612666,Mutation_Disease,DOID:824,Mutation,implicated,Disease,HALD_KG,Aging,dbSNP,DOID,rs4612666,Periodontitis
1,rs1143634,Mutation_Disease,DOID:824,Mutation,implicated,Disease,HALD_KG,Aging,dbSNP,DOID,rs1143634,Periodontitis
2,rs2619112,Mutation_Disease,D050723,Mutation,associated,Disease,HALD_KG,Aging,dbSNP,MESH,rs2619112,"Fractures, Bone"
3,rs2942168,Mutation_Disease,D010302,Mutation,associated,Disease,HALD_KG,Aging,dbSNP,MESH,rs2942168,"Parkinson Disease, Secondary"
4,rs2942168,Mutation_Disease,D010300,Mutation,associated,Disease,HALD_KG,Aging,dbSNP,MESH,rs2942168,Parkinson Disease
...,...,...,...,...,...,...,...,...,...,...,...,...
14829,LDLR_p.His583Asp,Mutation_Disease,C0745103,Mutation,associated_with_increased_likelihood_of,Disease,infores:clingen,Generalised,dbSNP,MESH,NM_000527.5(LDLR):c.1747C>G (p.His583Asp),"Hypercholesterolemia, familial, 1"
14830,LDLR_p.His583Arg,Mutation_Disease,C0745103,Mutation,associated_with_increased_likelihood_of,Disease,infores:clingen,Generalised,dbSNP,MESH,NM_000527.5(LDLR):c.1748A>G (p.His583Arg),"Hypercholesterolemia, familial, 1"
14831,LDLR_p.His583Gln,Mutation_Disease,C0745103,Mutation,genetically_associated_with,Disease,infores:clingen,Generalised,dbSNP,MESH,NM_000527.5(LDLR):c.1749C>G (p.His583Gln),"Hypercholesterolemia, familial, 1"
14832,LDLR_p.His583Gln,Mutation_Disease,C0745103,Mutation,associated_with_increased_likelihood_of,Disease,infores:clingen,Generalised,dbSNP,MESH,NM_000527.5(LDLR):c.1749C>A (p.His583Gln),"Hypercholesterolemia, familial, 1"


## 3 · NA Audit Before Deduplication

In [13]:
# Drop rows missing critical detail names
final_df = final_df.dropna(subset=['head_detail_name', 'tail_detail_name']).reset_index(drop=True)

print(f"Consolidated rows (before deduplication): {len(final_df):,}")
print("\nNaN audit before deduplication:")
print(final_df.isna().sum())

Consolidated rows (before deduplication): 14,834

NaN audit before deduplication:
head                0
relation            0
tail                0
head_type           0
relation_type       0
tail_type           0
kg_source           0
kg_type             0
head_id_is          0
tail_id_is          0
head_detail_name    0
tail_detail_name    0
dtype: int64


## 4 · Deduplication & NA Audit After Deduplication

In [14]:
def merge_sources(x):
    return '::'.join(sorted(set(x.dropna())))
group_cols = ['head', 'relation', 'tail']
final_df_dedup = final_df.groupby(group_cols, as_index=False).agg({
    'head_type': 'first', 'relation_type': 'first', 'tail_type': 'first',
    'kg_source': merge_sources, 'kg_type': merge_sources,
    'head_id_is': 'first', 'tail_id_is': 'first',
    'head_detail_name': 'first', 'tail_detail_name': 'first'
})

print(f"Before dedup: {len(final_df):,}  |  After dedup: {len(final_df_dedup):,}")
print("\nNaN audit after deduplication:")
print(final_df_dedup.isna().sum())

Before dedup: 14,834  |  After dedup: 14,685

NaN audit after deduplication:
head                0
relation            0
tail                0
head_type           0
relation_type       0
tail_type           0
kg_source           0
kg_type             0
head_id_is          0
tail_id_is          0
head_detail_name    0
tail_detail_name    0
dtype: int64


## 5 · Save Output

In [15]:
import os
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
final_df_dedup.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_df_dedup):,} rows → {OUT_PATH}")

Saved 14,685 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/MUTATION_DISEASE/ALL_MUTATION_DISEASE.csv


# Final

In [2]:
# =========================================================
# LOAD DISEASE MAPPING FILE
# =========================================================

DB_DIR = BASE_DIR + 'data_collection/databases_for_mapping/'

final_file_CH_D = pd.read_csv(
    DB_DIR + 'DO/ultimate_DISEASE_dictionary.csv'
)

# =========================================================
# PRIORITIZE DOID IDs
# =========================================================

# rows having DOID
doid_rows = final_file_CH_D[
    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)
]

# entity names that have at least one DOID
names_with_doid = set(
    doid_rows['entity_name']
    .str.lower()
    .str.strip()
)

# keep:
# 1. all DOID rows
# 2. non-DOID rows only if no DOID exists

final_diseaseid_df = final_file_CH_D[

    final_file_CH_D['entity_id']
    .str.startswith('DOID:', na=False)

    |

    ~final_file_CH_D['entity_name']
    .str.lower()
    .str.strip()
    .isin(names_with_doid)
]

final_diseaseid_df = (
    final_diseaseid_df
    .reset_index(drop=True)
)

# =========================================================
# CREATE DISEASE NAME -> ID DICTIONARY
# =========================================================

disease_dict = dict(

    zip(

        final_diseaseid_df['entity_name']
        .str.lower()
        .str.strip(),

        final_diseaseid_df['entity_id']
    )
)

# =========================================================
# UNIVERSAL ENTITY MAPPING FUNCTION
# =========================================================

def map_entity_ids(
    df,
    mapping_dict,
    side='tail'
):
    """
    Universal KG entity mapper.

    Parameters
    ----------
    df : pd.DataFrame

    mapping_dict : dict
        entity_name -> entity_id

    side : str
        'head' or 'tail'

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------------------
    # dynamic column names
    # -----------------------------------------

    detail_col = f'{side}_detail_name'

    id_col = side

    id_is_col = f'{side}_id_is'

    # -----------------------------------------
    # map IDs
    # -----------------------------------------

    df[id_col] = (df[detail_col].astype(str).str.lower().str.strip().map(mapping_dict))

    # -----------------------------------------
    # assign ID source
    # -----------------------------------------

    df[id_is_col] = None

    # DOID
    # DOID
    df.loc[
        df[id_col].str.startswith('DOID:', na=False),id_is_col] = 'DOID'
    
    # everything else -> MESH
    df.loc[~df[id_col].str.startswith('DOID:', na=False)&df[id_col].notna(),id_is_col] = 'MESH'

    return df

In [3]:
final_file = pd.read_csv(OUT_PATH)

final_file = map_entity_ids(df=final_file,mapping_dict=disease_dict,side='tail')

final_file

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,head_species,tail_species,species
0,ACADVL_p.Ala141fs,Mutation_Disease,DOID:0080155,Mutation,associated_with_increased_likelihood_of,Disease,infores:clinvar,Generalised,dbSNP,DOID,NM_000018.4(ACADVL):c.421dup (p.Ala141fs),very long chain acyl-CoA dehydrogenase deficiency,Homo sapiens,Homo sapiens,Homo sapiens
1,ACADVL_p.Ala161Thr,Mutation_Disease,DOID:0080155,Mutation,associated_with_increased_likelihood_of,Disease,infores:clingen,Generalised,dbSNP,DOID,NM_000018.4(ACADVL):c.481G>A (p.Ala161Thr),very long chain acyl-CoA dehydrogenase deficiency,Homo sapiens,Homo sapiens,Homo sapiens
2,ACADVL_p.Ala161Val,Mutation_Disease,DOID:0080155,Mutation,genetically_associated_with,Disease,infores:clingen,Generalised,dbSNP,DOID,NM_000018.4(ACADVL):c.482C>T (p.Ala161Val),very long chain acyl-CoA dehydrogenase deficiency,Homo sapiens,Homo sapiens,Homo sapiens
3,ACADVL_p.Ala180Thr,Mutation_Disease,DOID:0080155,Mutation,associated_with_increased_likelihood_of,Disease,infores:clinvar,Generalised,dbSNP,DOID,NM_000018.4(ACADVL):c.538G>A (p.Ala180Thr),very long chain acyl-CoA dehydrogenase deficiency,Homo sapiens,Homo sapiens,Homo sapiens
4,ACADVL_p.Ala213Pro,Mutation_Disease,DOID:0080155,Mutation,associated_with_increased_likelihood_of,Disease,infores:clinvar,Generalised,dbSNP,DOID,NM_000018.4(ACADVL):c.637G>C (p.Ala213Pro),very long chain acyl-CoA dehydrogenase deficiency,Homo sapiens,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14680,rs9939609,Mutation_Disease,D003643,Mutation,associated,Disease,HALD_KG,Aging,dbSNP,MESH,rs9939609,Death,Homo sapiens,Homo sapiens,Homo sapiens
14681,rs9939609,Mutation_Disease,D015430,Mutation,occur,Disease,HALD_KG,Aging,dbSNP,MESH,rs9939609,Weight Gain,Homo sapiens,Homo sapiens,Homo sapiens
14682,rs9939609,Mutation_Disease,C0872084,Mutation,associated,Disease,HALD_KG,Aging,dbSNP,MESH,rs9939609,Sarcopenia,Homo sapiens,Homo sapiens,Homo sapiens
14683,rs9939609,Mutation_Disease,DOID:9970,Mutation,associated,Disease,HALD_KG,Aging,dbSNP,DOID,rs9939609,Obesity,Homo sapiens,Homo sapiens,Homo sapiens


In [5]:
final_file[final_file['tail'].isna()]

,head,relation,tail,head_type,relation_type,tail_type,kg_source,kg_type,head_id_is,tail_id_is,head_detail_name,tail_detail_name,head_species,tail_species,species


In [6]:
final_file.to_csv(OUT_PATH, index=False)
print(f"Saved {len(final_file):,} rows → {OUT_PATH}")

Saved 14,685 rows → /storage/Arushi/090526_EvoAge/kg_formation/processed_data_relation_wise_merge/generalised/MUTATION_DISEASE/ALL_MUTATION_DISEASE.csv
